
# Proyecto Final Parte 2 - Reproductor MIDI

In [14]:
#@title Preparacion: Instalar sintetizador MIDI
!apt-get -q -qq install -y fluidsynth fluid-soundfont-gm

In [2]:
#@title Preparacion: Instalación bibliotecas

# Instalar las bibliotecas de Python
!pip -q install kagglehub pretty_midi ipywidgets soundfile matplotlib midi2audio

In [9]:
#@title Preparacion: Importación de librerias

import os, json, html, base64, tempfile, subprocess
import io
from pathlib import Path
from google.colab import files, output
import numpy as np
import soundfile as sf
import pretty_midi
import ipywidgets as widgets
from IPython.display import display, Audio, HTML, clear_output
from midi2audio import FluidSynth

In [4]:
#@title 0. Dataset

# Se define la ruta de la carpeta en Colab
ruta_carpeta = Path('/content/MIDIS') if Path('/content/MIDIS').exists() else Path('examples')

# Buscamos automáticamente todos los archivos con extensión .mid o .midi dentro de esa carpeta
archivos_midi = sorted(list(ruta_carpeta.glob('*.mid')))

# Definir la variable 'datasets' mapeada para la interfaz
datasets = {
    'Mis Canciones': archivos_midi
}

# Definir la carpeta de resultados
ruta_resultados = "/content/MIDIS"

In [38]:
#@title 1. Interfaz

# Banco de sonidos MIDI
SOUNDFONT = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

# 1. Lógica dinámica para listar canciones
def obtener_canciones():
    if not os.path.exists(ruta_resultados):
        os.makedirs(ruta_resultados, exist_ok=True)
    canciones = [archivo for archivo in os.listdir(ruta_resultados) if archivo.endswith(('.mid', '.midi'))]
    return canciones if canciones else ["No se encontraron archivos MIDI"]

selector_cancion = widgets.Dropdown(
    options=obtener_canciones(),
    description='Canción:',
    style={'description_width': 'initial'}
)

boton_actualizar = widgets.Button(description="🔄 Actualizar Lista", button_style='info')

def refrescar_opciones(b):
    selector_cancion.options = obtener_canciones()

boton_actualizar.on_click(refrescar_opciones)

# 2. Recrear el selector de instrumentos y el checkbox
lista_instrumentos = [
    ("0 - Piano de Cola Acústico", 0), ("1 - Piano Acústico Brillante", 1), ("2 - Piano de Cola Eléctrico", 2),
    ("3 - Piano Honky-tonk", 3), ("4 - Piano Eléctrico 1", 4), ("5 - Piano Eléctrico 2", 5),
    ("6 - Clavicémbalo", 6), ("7 - Clavinet", 7), ("8 - Celesta", 8), ("9 - Glockenspiel", 9),
    ("10 - Caja de Música", 10), ("11 - Vibráfono", 11), ("12 - Marimba", 12), ("13 - Xilófono", 13),
    ("14 - Campanas Tubulares", 14), ("15 - Dulcémele", 15), ("16 - Órgano de Barras", 16),
    ("17 - Órgano Percusivo", 17), ("18 - Órgano de Rock", 18), ("19 - Órgano de Iglesia", 19),
    ("20 - Órgano de Lengüeta", 20), ("21 - Acordeón", 21), ("22 - Armónica", 22),
    ("23 - Acordeón de Tango", 23), ("24 - Guitarra Acústica (nailon)", 24), ("25 - Guitarra Acústica (acero)", 25),
    ("26 - Guitarra Eléctrica (jazz)", 26), ("27 - Guitarra Eléctrica (limpia)", 27), ("28 - Guitarra Eléctrica (sordina)", 28),
    ("29 - Guitarra con Sobresaturación", 29), ("30 - Guitarra de Distorsión", 30), ("31 - Armónicos de Guitarra", 31),
    ("32 - Bajo Acústico", 32), ("33 - Bajo Eléctrico (dedos)", 33), ("34 - Bajo Eléctrico (púa)", 34),
    ("35 - Bajo sin Trastes", 35), ("36 - Bajo Slap 1", 36), ("37 - Bajo Slap 2", 37),
    ("38 - Bajo Sintetizado 1", 38), ("39 - Bajo Sintetizado 2", 39), ("40 - Violín", 40),
    ("41 - Viola", 41), ("42 - Violonchelo", 42), ("43 - Contrabajo", 43), ("44 - Cuerdas Trémolo", 44),
    ("45 - Cuerdas Pizzicato", 45), ("46 - Arpa Orquestal", 46), ("47 - Timbales", 47),
    ("48 - Ensamble de Cuerdas 1", 48), ("49 - Ensamble de Cuerdas 2", 49), ("50 - Cuerdas Sintetizadas 1", 50),
    ("51 - Cuerdas Sintetizadas 2", 51), ("52 - Coro de Aaahs", 52), ("53 - Voces de Ooohs", 53),
    ("54 - Coro Sintetizado", 54), ("55 - Golpe de Orquesta", 55), ("56 - Trompeta", 56),
    ("57 - Trombón", 57), ("58 - Tuba", 58), ("59 - Trompeta con Sordina", 59), ("60 - Trompa (Corno Francés)", 60),
    ("61 - Sección de Metales", 61), ("62 - Metales Sintetizados 1", 62), ("63 - Metales Sintetizados 2", 63),
    ("64 - Saxofón Soprano", 64), ("65 - Saxofón Alto", 65), ("66 - Saxofón Tenor", 66), ("67 - Saxofón Barítono", 67),
    ("68 - Oboe", 68), ("69 - Corno Inglés", 69), ("70 - Fagot", 70), ("71 - Clarinete", 71),
    ("72 - Flautín", 72), ("73 - Flauta", 73), ("74 - Flauta Dulce", 74), ("75 - Flauta de Pan", 75),
    ("76 - Botella Soplada", 76), ("77 - Shakuhachi", 77), ("78 - Silbato", 78), ("79 - Ocarina", 79),
    ("80 - Sonido Guía 1 (onda cuadrada)", 80), ("81 - Sonido Guía 2 (onda de sierra)", 81), ("82 - Sonido Guía 3 (calíope)", 82),
    ("83 - Sonido Guía 4 (chiff)", 83), ("84 - Sonido Guía 5 (charang)", 84), ("85 - Sonido Guía 6 (voz)", 85),
    ("86 - Sonido Guía 7 (quintas)", 86), ("87 - Sonido Guía 8 (bajo + guía)", 87), ("88 - Fondo 1 (new age)", 88),
    ("89 - Fondo 2 (cálido)", 89), ("90 - Fondo 3 (polysynth)", 90), ("91 - Fondo 4 (coro)", 91),
    ("92 - Fondo 5 (arco)", 92), ("93 - Fondo 6 (metálico)", 93), ("94 - Fondo 7 (halo)", 94),
    ("95 - Fondo 8 (barrido)", 95), ("96 - Efecto de Sonido 1 (lluvia)", 96), ("97 - Efecto de Sonido 2 (banda sonora)", 97),
    ("98 - Efecto de Sonido 3 (cristal)", 98), ("99 - Efecto de Sonido 4 (atmósfera)", 99), ("100 - Efecto de Sonido 5 (brillo)", 100),
    ("101 - Efecto de Sonido 6 (goblins)", 101), ("102 - Efecto de Sonido 7 (ecos)", 102), ("103 - Efecto de Sonido 8 (ciencia ficción)", 103),
    ("104 - Sitar", 104), ("105 - Banjo", 105), ("106 - Shamisen", 106), ("107 - Koto", 107),
    ("108 - Kalimba", 108), ("109 - Gaita", 109), ("110 - Violín de música folclórica", 110), ("111 - Shanai", 111),
    ("112 - Campanilla", 112), ("113 - Agogô", 113), ("114 - Tambor de Acero", 114),
    ("115 - Bloque de Madera", 115), ("116 - Tambor Taiko", 116), ("117 - Tom Melódico", 117),
    ("118 - Tambor Sintetizado", 118), ("119 - Platillo Invertido", 119), ("120 - Ruido de Traste de Guitarra", 120),
    ("121 - Ruido de Respiración", 121), ("122 - Costa Marina", 122), ("123 - Canto de Pájaro", 123),
    ("124 - Timbre de Teléfono", 124), ("125 - Helicóptero", 125), ("126 - Aplausos", 126),
    ("127 - Disparo de Arma", 127)
]

selector_instrumento = widgets.Dropdown(
    options=lista_instrumentos,
    value=0,
    description='Instrumento:',
    style={'description_width': 'initial'}
)

checkbox_original = widgets.Checkbox(
    value=False,
    description='Conservar instrumentos originales',
    indent=False
)

# 3. Botón de reproducción y área de salida
boton_reproducir = widgets.Button(description="▶ Reproducir", button_style='success')
salida_audio = widgets.Output()

# Interfaz agrupada
ui_superior = widgets.HBox([selector_cancion, boton_actualizar])
ui_media = widgets.HBox([selector_instrumento, checkbox_original])
display(widgets.VBox([ui_superior, ui_media, boton_reproducir, salida_audio]))

# 4. Lógica central de procesamiento y reproducción
def procesar_y_reproducir(b):
    with salida_audio:
        clear_output(wait=True)
        cancion_seleccionada = selector_cancion.value

        if cancion_seleccionada == "No se encontraron archivos MIDI":
            print("Por favor, genera o agrega una canción primero.")
            return

        ruta_entrada = os.path.join(ruta_resultados, cancion_seleccionada)
        ruta_wav = ruta_entrada.replace('.mid', '.wav').replace('.midi', '.wav')

        try:
            print("Preparando audio...")

            # Si NO queremos los originales, sobrescribimos el instrumento
            if not checkbox_original.value:
                midi_data = pretty_midi.PrettyMIDI(ruta_entrada)
                instrumento_nuevo = selector_instrumento.value

                for instrumento in midi_data.instruments:
                    if not instrumento.is_drum:
                        instrumento.program = instrumento_nuevo

                # Guardamos un archivo temporal modificado
                ruta_entrada = os.path.join(ruta_resultados, "temp_modificado.mid")
                midi_data.write(ruta_entrada)

            # Convertir a WAV usando FluidSynth (asegúrate de tener instalado el soundfont)
            fs = FluidSynth()
            fs.midi_to_audio(ruta_entrada, ruta_wav)

            # Mostrar el reproductor
            print("¡Listo para escuchar!")
            display(Audio(ruta_wav))

        except Exception as e:
            print(f"Ocurrió un error al procesar el archivo: {e}")

boton_reproducir.on_click(procesar_y_reproducir)

In [47]:
#@title 2. Carga de MIDI

output.enable_custom_widget_manager()

# Variables globales para guardar la selección confirmada
cancion_confirmada = None
instrumento_confirmado = None
conservar_original_confirmado = False

# 2. Crear interfaz
def obtener_archivos():
    if not os.path.exists(ruta_resultados):
        os.makedirs(ruta_resultados, exist_ok=True)
    archivos_encontrados = [f for f in os.listdir(ruta_resultados) if f.endswith(('.mid', '.midi'))]
    return archivos_encontrados if archivos_encontrados else ["No se encontraron archivos"]

# 1. Tu código de interfaz, adaptado para ser dinámico
cancion = widgets.Dropdown(
    options=obtener_archivos(),
    description="Canción:",
    layout=widgets.Layout(width="95%")
)

instrumento = widgets.Dropdown(
    options=[("Original", -1)] + [(lista_instrumentos[i]) for i in range(128)],
    value=0,
    description="Instrumento:",
    layout=widgets.Layout(width="95%")
)

checkbox_original = widgets.Checkbox(
    value=False,
    description='Conservar instrumentos originales',
    indent=False
)

boton_actualizar = widgets.Button(description="ℱ Actualizar Lista", button_style='info')
boton_confirmar = widgets.Button(description="✅ Confirmar Selección", button_style='success', layout=widgets.Layout(width="95%"))
salida_confirmacion = widgets.Output()

# 2. Lógica para refrescar la lista de canciones
def refrescar_canciones(b):
    cancion.options = obtener_archivos()

boton_actualizar.on_click(refrescar_canciones)

# Lógica para confirmar selección
def confirmar_seleccion(b):
    global cancion_confirmada, instrumento_confirmado, conservar_original_confirmado
    cancion_confirmada = cancion.value
    instrumento_confirmado = instrumento.value
    conservar_original_confirmado = checkbox_original.value

    with salida_confirmacion:
        clear_output()
        if cancion_confirmada == "No se encontraron archivos":
            print("❌ No hay ninguna canción válida seleccionada.")
        else:
            inst_nombre = "Original" if instrumento_confirmado == -1 else lista_instrumentos[instrumento_confirmado][0]
            print(f"🎵 Configuración Guardada:\n• Canción: {cancion_confirmada}\n• Instrumento: {inst_nombre}\n• Mantener Originales: {'Sí' if conservar_original_confirmado else 'No'}")

boton_confirmar.on_click(confirmar_seleccion)

vi_superior = widgets.HBox([cancion, boton_actualizar])
vi_media = widgets.HBox([instrumento, checkbox_original])
display(widgets.VBox([vi_superior, vi_media, boton_confirmar, salida_confirmacion]))

In [50]:
#@title 3. Mostrar Reproductor

# Crear un área de salida dinámica para el procesamiento y audio
salida_reproductor = widgets.Output()
boton_procesar_confirmado = widgets.Button(
    description="▶ Procesar y Escuchar Selección Confirmada",
    button_style='success',
    layout=widgets.Layout(width="95%")
)

def reproducir_confirmado(b):
    with salida_reproductor:
        clear_output(wait=True)

        # Validamos si ya confirmaron opciones en la celda anterior
        if cancion_confirmada is None:
            print("❌ Primero debes hacer clic en 'Confirmar Selección' en la celda de arriba.")
            return

        try:
          conser_orig = "se conservaron instrumentos originales" if conservar_original_confirmado == "Si" else "no se conservaron instrumentos originales"
        except Exception as e:
          print("Selecciona una opcion valida")

        ruta_entrada = os.path.join(ruta_resultados, cancion_confirmada)
        ruta_wav = ruta_entrada.replace('.mid', '.wav').replace('.midi', '.wav')

        try:
            if instrumento_confirmado == -1:
                inst_nombre = "Original"
            else:
                # Obtener el nombre completo y quitarle el prefijo numérico (ej. "24 - Guitarra" -> "Guitarra")
                nombre_completo = lista_instrumentos[instrumento_confirmado][0]
                if " - " in nombre_completo:
                    inst_nombre = nombre_completo.split(" - ", 1)[1]
                else:
                    inst_nombre = nombre_completo

            print(f"🎵 Procesando canción: {cancion_confirmada} con {inst_nombre} y {conser_orig}...")

            # Si no se conservan originales, aplicamos el instrumento confirmado
            if not conservar_original_confirmado:
                midi_data = pretty_midi.PrettyMIDI(ruta_entrada)

                # Si instrumento_confirmado es distinto de -1, lo aplicamos
                if instrumento_confirmado != -1:
                    for inst in midi_data.instruments:
                        if not inst.is_drum:
                            inst.program = instrumento_confirmado

                ruta_entrada = os.path.join(ruta_resultados, "temp_modificado.mid")
                midi_data.write(ruta_entrada)

            # Convertir a WAV con FluidSynth
            fs = FluidSynth()
            fs.midi_to_audio(ruta_entrada, ruta_wav)

            print("¡Listo para reproducir!")
            display(Audio(ruta_wav))

        except Exception as e:
            print(f"❌ Ocurrió un error: {e}")

boton_procesar_confirmado.on_click(reproducir_confirmado)

display(widgets.VBox([
    widgets.HTML("<h2>🎹 MIDI Studio</h2>"),
    widgets.HTML("<b>Reproducción de Selección Confirmada</b>"),
    boton_procesar_confirmado,
    salida_reproductor
], layout=widgets.Layout(
    border="1px solid #50658a",
    padding="18px",
    width="96%"
)))